In [ ]:
# =============================================================================
# QAT lambda=0 reanalysis, calibration confound, agreement conflict,
# =============================================================================
import os, sys, glob, json, math, time, zipfile, warnings, shutil
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
#
T_START = time.time()
OUT = '/kaggle/working/rev3'
if not os.path.isdir('/kaggle/working'):
    OUT = '/tmp/rev3'
os.makedirs(OUT, exist_ok=True)
STATUS = []
ARCHS = ['tf_efficientnetv2_s', 'resnet50', 'mobilenetv3_large_100']
XAIS = ['gradcam', 'gradcampp', 'ig']
K_MAIN = 0.15
SEED = 42
IMG = 224
MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD = np.array([0.229, 0.224, 0.225], np.float32)
#
#
def log(msg):
    print('[%7.1fs] %s' % (time.time() - T_START, msg), flush=True)
#
#
def record(task, state, detail):
    STATUS.append(dict(task=task, status=state, detail=str(detail)[:300]))
    log('%-28s %-8s %s' % (task, state, str(detail)[:150]))
#
#
def save(name, df):
    p = os.path.join(OUT, name + '.csv')
    df.to_csv(p, index=False)
    log('   saved %s  (%d rows)' % (p, len(df)))
    return p
#
#
# ---------------------------------------------------------------------------
# 0. INPUT DISCOVERY (zip-aware, layout-agnostic)
# ---------------------------------------------------------------------------
ORT_OK = False
try:
    import onnxruntime as _p
    ORT_OK = True
except Exception:
    for _spec in ['onnxruntime==1.19.2', 'onnxruntime']:
        os.system(sys.executable + ' -m pip install -q ' + _spec + ' > /dev/null 2>&1')
        try:
            import onnxruntime as _p
            ORT_OK = True
            break
        except Exception:
            continue
try:
    import onnx as _q
except Exception:
    os.system(sys.executable + ' -m pip install -q onnx > /dev/null 2>&1')
print('onnxruntime available:', ORT_OK, flush=True)


KNOWN_TABLES = '/kaggle/input/datasets/sunzilkhandaker/quantxai-endgame/tables'
KNOWN_PADDY = ('/kaggle/input/datasets/imbikramsaha/paddy-doctor/'
               'paddy-disease-classification/train_images')
#
ROOTS = [p for p in ['/kaggle/input', '/kaggle/working', '/data'] if os.path.isdir(p)]
log('search roots: %s' % ROOTS)
#
#
def find(pattern, roots=None, limit=None):
    hits = []
    if roots is None and os.path.isdir(KNOWN_TABLES):
        fast = sorted(glob.glob(os.path.join(KNOWN_TABLES, pattern)))
        if fast:
            return fast[:limit] if limit else fast
    for r in (roots or ROOTS):
        hits.extend(glob.glob(os.path.join(r, '**', pattern), recursive=True))
    hits = sorted(set(hits))
    return hits[:limit] if limit else hits

# auto-extract nested zips only if the tables are not already visible
if not find('RAW_qat_drift.csv'):
    for z in find('*.zip')[:8]:
        try:
            d = '/kaggle/working/_unz/' + os.path.splitext(os.path.basename(z))[0]
            if os.path.isdir(d):
                continue
            os.makedirs(d, exist_ok=True)
            with zipfile.ZipFile(z) as zf:
                zf.extractall(d)
            log('extracted %s -> %s' % (os.path.basename(z), d))
        except Exception as e:
            log('zip skip %s (%s)' % (os.path.basename(z), type(e).__name__))
    ROOTS = list(dict.fromkeys(ROOTS + ['/kaggle/working/_unz']))
#
#
def first(pattern):
    h = find(pattern)
    return h[0] if h else None
#
#
P_QAT = first('RAW_qat_drift.csv')
P_DRIFT = first('RAW_drift_all.csv')
P_T8 = first('T8_qat_mitigation.csv')
P_N10 = first('N10_lambda_sweep.csv')
P_N3 = first('N3_calibration_ablation.csv')
P_T5 = first('T5_prediction_agreement.csv')
log('RAW_qat_drift      : %s' % P_QAT)
log('RAW_drift_all      : %s' % P_DRIFT)
log('T8_qat_mitigation  : %s' % P_T8)
log('N10_lambda_sweep   : %s' % P_N10)
log('N3_calib_ablation  : %s' % P_N3)
log('T5_pred_agreement  : %s' % P_T5)

# ---------------------------------------------------------------------------
# STATISTICS (pure numpy - no scipy dependency, identical to scipy defaults)
# ---------------------------------------------------------------------------
def _rankdata(a):
    a = np.asarray(a, float)
    order = np.argsort(a, kind='mergesort')
    sa = a[order]
    ranks = np.empty(a.size, float)
    i = 0
    while i < a.size:
        j = i
        while j + 1 < a.size and sa[j + 1] == sa[i]:
            j += 1
        ranks[order[i:j + 1]] = 0.5 * (i + j) + 1.0
        i = j + 1
    return ranks
#
#
def _norm_sf(z):
    return 0.5 * math.erfc(float(z) / math.sqrt(2.0))
#
#
def wilcoxon_signed_rank(d):
    """Two-sided Wilcoxon signed-rank, normal approx w/ tie + continuity correction."""
    d = np.asarray(d, float)
    d = d[np.isfinite(d)]
    d = d[d != 0.0]
    n = int(d.size)
    if n < 1:
        return float('nan'), 1.0, 0
    r = _rankdata(np.abs(d))
    wp = float(r[d > 0].sum())
    wm = float(r[d < 0].sum())
    W = min(wp, wm)
    mu = n * (n + 1.0) / 4.0
    _, cnt = np.unique(np.abs(d), return_counts=True)
    tie = float(((cnt.astype(float) ** 3) - cnt.astype(float)).sum())
    var = n * (n + 1.0) * (2.0 * n + 1.0) / 24.0 - tie / 48.0
    if var <= 0:
        return W, 1.0, n
    z = (W - mu + 0.5) / math.sqrt(var)
    return W, float(min(1.0, 2.0 * _norm_sf(abs(z)))), n
#
#
def holm(ps):
    ps = np.asarray(ps, float)
    m = ps.size
    order = np.argsort(ps)
    adj = np.empty(m, float)
    run = 0.0
    for i, idx in enumerate(order):
        run = max(run, (m - i) * ps[idx])
        adj[idx] = min(1.0, run)
    return adj
#
#
def cliffs_delta(x, y):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    gt = 0
    lt = 0
    for v in x:
        gt += int((v > y).sum())
        lt += int((v < y).sum())
    n = x.size * y.size
    return (gt - lt) / n if n else float('nan')


PADDY = None
VAL_DF = None
TRAIN_DF = None
EVAL320 = None
SPLIT_OK = False
try:
    t0 = time.time()
    from sklearn.model_selection import train_test_split
    train_dir = KNOWN_PADDY if os.path.isdir(KNOWN_PADDY) else None
    if train_dir is None:
        cand = []
        for r in ROOTS:
            for sub in glob.glob(os.path.join(r, '*')):
                if 'paddy' in os.path.basename(sub).lower():
                    cand.append(sub)
        for c in (cand or ROOTS):
            for probe in ['train_images', 'train', 'images']:
                for h in glob.glob(os.path.join(c, '**', probe), recursive=True):
                    if not os.path.isdir(h):
                        continue
                    subs = [d for d in glob.glob(os.path.join(h, '*')) if os.path.isdir(d)]
                    if len(subs) >= 5:
                        train_dir = h
                        break
                if train_dir:
                    break
            if train_dir:
                break
    assert train_dir, 'paddy class-folder directory not found'
    recs = []
    for cd in sorted(glob.glob(os.path.join(train_dir, '*'))):
        if not os.path.isdir(cd):
            continue
        lab = os.path.basename(cd)
        for ip in glob.glob(os.path.join(cd, '*')):
            if os.path.splitext(ip)[1].lower() in ('.jpg', '.jpeg', '.png', '.bmp'):
                recs.append(dict(path=ip, image=os.path.basename(ip), label=lab))
    PADDY = pd.DataFrame(recs).sort_values('path').reset_index(drop=True)
    classes = sorted(PADDY.label.unique().tolist())
    PADDY['y'] = PADDY.label.map({c: i for i, c in enumerate(classes)})
    log('paddy: %d images / %d classes @ %s' % (len(PADDY), len(classes), train_dir))
    if len(PADDY) != 10407 or len(classes) != 10:
        log('   !! WARNING expected 10407 images / 10 classes - split will NOT match')
    tr_idx, va_idx = train_test_split(np.arange(len(PADDY)), test_size=0.20,
                                      random_state=SEED, stratify=PADDY.y.values)
    VAL_DF = PADDY.iloc[va_idx].reset_index(drop=True)
    TRAIN_DF = PADDY.iloc[tr_idx].reset_index(drop=True)
    log('split: train=%d val=%d  (paper: 8325 / 2082)' % (len(TRAIN_DF), len(VAL_DF)))
    assert P_DRIFT is not None, 'RAW_drift_all.csv not found; cannot verify the split'
    EVAL320 = sorted(pd.read_csv(P_DRIFT).image.unique().tolist())
    nva = len(set(EVAL320) & set(VAL_DF.image))
    ntr = len(set(EVAL320) & set(TRAIN_DF.image))
    log('CONTAINMENT GATE: %d/%d published eval images in val, %d leaked into train'
        % (nva, len(EVAL320), ntr))
    assert nva == len(EVAL320), ('HARD GATE FAILED: only %d/%d published eval images '
                                 'fall in the rebuilt val split, so any full-split '
                                 'number would be computed partly on TRAINING data. '
                                 'Refusing to run D and E.' % (nva, len(EVAL320)))
    assert ntr == 0, 'HARD GATE FAILED: %d eval images leaked into train' % ntr
    SPLIT_OK = True
    log('   gate PASSED - val split reproduces the original run exactly')
    record('SPLIT_reconstruction', 'OK',
           'paddy=%d val=%d train=%d containment=%d/320 (%.1fs)'
           % (len(PADDY), len(VAL_DF), len(TRAIN_DF), nva, time.time() - t0))
except Exception as e:
    record('SPLIT_reconstruction', 'FAILED', '%s: %s' % (type(e).__name__, e))
#
#
def load_batch(paths, size=IMG):
    from PIL import Image
    arr = np.zeros((len(paths), 3, size, size), np.float32)
    for i, p in enumerate(paths):
        im = Image.open(p).convert('RGB').resize((size, size), Image.BILINEAR)
        a = np.asarray(im, np.float32) / 255.0
        arr[i] = ((a - MEAN) / STD).transpose(2, 0, 1)
    return arr
#
#
# ===========================================================================
# ORT calibration ablation at a MATCHED n=64 cap
# Closes BLOCKER 2. All three calibrators share one fixed 64-image subset.
# ===========================================================================
try:
    t0 = time.time()
    import onnxruntime as ort
    from onnxruntime.quantization import (quantize_static, CalibrationDataReader,
                                          CalibrationMethod, QuantFormat, QuantType)
    assert PADDY is not None and SPLIT_OK, 'paddy dataset/split unavailable'
    fp32_map = {}
    for a in ARCHS:
        for pat in ['*%s*fp32*.onnx' % a, '*%s*.onnx' % a]:
            h = [x for x in find(pat) if 'int8' not in os.path.basename(x).lower()]
            if h:
                fp32_map[a] = h[0]
                break
    assert fp32_map, 'no FP32 .onnx graphs found in input'
    log('fp32 graphs: %s' % {k: os.path.basename(v) for k, v in fp32_map.items()})
    # fixed, seeded, stratified 64-image calibration subset from TRAIN
    rng = np.random.RandomState(SEED)
    per_cls = max(1, 64 // PADDY.label.nunique())
    cal_idx = []
    for c in sorted(TRAIN_DF.label.unique()):
        idx = TRAIN_DF.index[TRAIN_DF.label == c].values.copy()
        rng.shuffle(idx)
        cal_idx.extend(idx[:per_cls].tolist())
    cal_idx = cal_idx[:64]
    CAL_DF = TRAIN_DF.loc[cal_idx].reset_index(drop=True)
    CAL_X = load_batch(CAL_DF.path.tolist())
    log('calibration tensor: %s (fixed for every method and architecture)' % (CAL_X.shape,))
    # exact published 320-image evaluation subset, resolved by filename
    if EVAL320:
        ev = PADDY[PADDY.image.isin(EVAL320)].reset_index(drop=True)
    else:
        ev = VAL_DF.sample(n=320, random_state=SEED).reset_index(drop=True)
    log('eval subset: %d images (exact published subset = %s)' % (len(ev), bool(EVAL320)))
    EV_X = load_batch(ev.path.tolist())
    EV_Y = ev.y.values
    #
    class DR(CalibrationDataReader):
        def __init__(self, arr, iname, bs=1):
            self.items = [{iname: arr[i:i + bs]} for i in range(0, len(arr), bs)]
            self.it = iter(self.items)
        def get_next(self):
            return next(self.it, None)
        def rewind(self):
            self.it = iter(self.items)
    #
    def run_sess(path, X, bs=16):
        so = ort.SessionOptions()
        so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        so.intra_op_num_threads = max(1, (os.cpu_count() or 2))
        s = ort.InferenceSession(path, sess_options=so, providers=['CPUExecutionProvider'])
        iname = s.get_inputs()[0].name
        outs = []
        for i in range(0, len(X), bs):
            outs.append(s.run(None, {iname: X[i:i + bs]})[0])
        return np.concatenate(outs, 0)
    #
    methods = [('MinMax', CalibrationMethod.MinMax, {}),
               ('Entropy', CalibrationMethod.Entropy, {}),
               ('Percentile', CalibrationMethod.Percentile, {'CalibPercentile': 99.999})]
    rows = []
    qdir = os.path.join(OUT, 'onnx_matched64')
    os.makedirs(qdir, exist_ok=True)
    def preprocess(fp):
        """quant_pre_process makes quantize_static robust on older opsets."""
        try:
            from onnxruntime.quantization.shape_inference import quant_pre_process
            pre = os.path.join(qdir, 'pre_' + os.path.basename(fp))
            if not os.path.exists(pre):
                quant_pre_process(fp, pre, skip_symbolic_shape=False)
            return pre
        except Exception as ep:
            log('   pre-process skipped (%s); using raw graph' % type(ep).__name__)
            return fp
    #
    for arch, fp in fp32_map.items():
        try:
            fp_pre = preprocess(fp)
            fp_logits = run_sess(fp, EV_X)
            fp_pred = fp_logits.argmax(1)
            fp_acc = float((fp_pred == EV_Y).mean())
            s0 = ort.InferenceSession(fp, providers=['CPUExecutionProvider'])
            iname = s0.get_inputs()[0].name
            for mname, mmeth, xopt in methods:
                try:
                    outp = os.path.join(qdir, '%s_int8_%s_n64.onnx' % (arch, mname.lower()))
                    if not os.path.exists(outp):
                        opts = {'ActivationSymmetric': False, 'WeightSymmetric': True}
                        opts.update(xopt)
                        quantize_static(fp_pre, outp, DR(CAL_X, iname),
                                        quant_format=QuantFormat.QDQ,
                                        per_channel=True,
                                        activation_type=QuantType.QUInt8,
                                        weight_type=QuantType.QInt8,
                                        calibrate_method=mmeth,
                                        extra_options=opts)
                    q_pred = run_sess(outp, EV_X).argmax(1)
                    rows.append(dict(arch=arch, engine='onnxruntime', calib_source='train_subset',
                                     calib_method=mname, n_calib=64, n_eval=len(ev),
                                     agreement=round(float((q_pred == fp_pred).mean()), 4),
                                     int8_acc=round(float((q_pred == EV_Y).mean()), 4),
                                     fp32_acc=round(fp_acc, 4),
                                     size_mb=round(os.path.getsize(outp) / 1e6, 2)))
                    log('   %-22s %-11s agreement=%.4f int8_acc=%.4f'
                        % (arch, mname, rows[-1]['agreement'], rows[-1]['int8_acc']))
                except Exception as e2:
                    log('   %-22s %-11s FAILED %s: %s' % (arch, mname, type(e2).__name__, e2))
        except Exception as e1:
            log('   %s skipped (%s: %s)' % (arch, type(e1).__name__, e1))
    assert rows, 'no matched-64 quantization succeeded'
    Dq = pd.DataFrame(rows)
    # fidelity gate: does the rebuild reproduce the published n=64 Entropy/Percentile?
    if P_N3:
        n3 = pd.read_csv(P_N3)
        pub = n3[(n3.engine == 'onnxruntime')][['arch', 'calib_method', 'n_calib', 'agreement', 'int8_acc']]
        pub = pub.rename(columns=dict(agreement='published_agreement', int8_acc='published_int8_acc',
                                      n_calib='published_n_calib'))
        Dq = Dq.merge(pub, on=['arch', 'calib_method'], how='left')
        Dq['delta_vs_published'] = (Dq.agreement - Dq.published_agreement).round(4)
        gate = Dq[Dq.published_n_calib == 64]
        worst = float(gate.delta_vs_published.abs().max()) if len(gate) else float('nan')
        Dq['rebuild_fidelity_maxabs_delta_at_matched_n'] = round(worst, 4) if worst == worst else float('nan')
        log('FIDELITY GATE: max |rebuilt - published| on the already-n=64 arms = %s' % worst)
    save('N25_calibration_matched64', Dq)
    record('D_calib_matched64', 'OK',
           '%d rows, matched n_calib=64 for all calibrators (%.1fs)' % (len(Dq), time.time() - t0))
    print(Dq.to_string(index=False))
except Exception as e:
    record('D_calib_matched64', 'SKIPPED', '%s: %s' % (type(e).__name__, e))
#
#
# ===========================================================================
# Grad-CAM + Grad-CAM++ on the FULL 2,082 val split
# Gated: first reproduces the published 320-subset value; only then extends.
# ===========================================================================
try:
    t0 = time.time()
    BUDGET_S = float(os.environ.get('REV3_TASK_E_BUDGET_S', 3600))
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    import timm
    assert PADDY is not None and SPLIT_OK, 'paddy dataset/split unavailable'
    DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
    assert DEV == 'cuda', 'no GPU attached; skipping full-split CAM (enable a T4 to run it)'
    ckpts = {}
    for a in ARCHS:
        h = find('*%s*.pt' % a) + find('*%s*.pth' % a)
        if h:
            ckpts[a] = h[0]
    assert ckpts, 'no .pt/.pth checkpoints found in input'
    log('checkpoints: %s' % {k: os.path.basename(v) for k, v in ckpts.items()})
    NCLS = int(PADDY.y.nunique())
    PREFERRED = {'resnet50': ['layer4', 'layer4.2'],
                 'tf_efficientnetv2_s': ['bn2', 'conv_head', 'blocks'],
                 'mobilenetv3_large_100': ['blocks', 'blocks.6']}
    ACT_T = (nn.ReLU, nn.ReLU6, nn.SiLU, nn.Hardswish, nn.GELU, nn.Hardsigmoid, nn.Hardtanh)
    #
    class WeightFQ(nn.Module):
        def __init__(self, qmax=127):
            super().__init__()
            self.qmax = qmax
        def forward(self, W):
            dims = tuple(range(1, W.dim()))
            s = torch.clamp(W.detach().abs().amax(dim=dims, keepdim=True) / self.qmax, min=1e-12)
            Wq = torch.clamp(torch.round(W / s), -self.qmax, self.qmax) * s
            return W + (Wq - W).detach()
    #
    class ActFQ(nn.Module):
        def __init__(self):
            super().__init__()
            self.register_buffer('mn', torch.tensor(float('inf')))
            self.register_buffer('mx', torch.tensor(float('-inf')))
            self.observing, self.enabled = True, True
        def forward(self, x):
            if not torch.is_tensor(x) or not x.is_floating_point():
                return x
            if self.observing:
                with torch.no_grad():
                    xd = x.detach().float()
                    self.mn = torch.minimum(self.mn, xd.min())
                    self.mx = torch.maximum(self.mx, xd.max())
                return x
            if not self.enabled or not torch.isfinite(self.mn) or not torch.isfinite(self.mx):
                return x
            mn = torch.clamp(self.mn, max=0.0).to(x.dtype)
            mx = torch.clamp(self.mx, min=0.0).to(x.dtype)
            scale = torch.clamp((mx - mn) / 255.0, min=1e-12)
            zp = torch.round(-mn / scale).clamp(0, 255)
            qv = torch.clamp(torch.round(x / scale) + zp, 0, 255)
            return x + ((qv - zp) * scale - x).detach()
    #
    def fold_conv_bn(model):
        import torch.nn.utils.fusion as fusion
        n = [0]
        def rec(mod):
            for _, ch in mod.named_children():
                rec(ch)
            if isinstance(mod, nn.Sequential):
                names = [k for k, _ in mod.named_children()]
                mods = list(mod.children())
                i = 0
                while i + 1 < len(mods):
                    a, b = mods[i], mods[i + 1]
                    if isinstance(a, nn.Conv2d) and type(b) is nn.BatchNorm2d:
                        setattr(mod, names[i], fusion.fuse_conv_bn_eval(a.eval(), b.eval()))
                        setattr(mod, names[i + 1], nn.Identity())
                        n[0] += 1
                        i += 2
                    else:
                        i += 1
            c, b = getattr(mod, 'conv', None), getattr(mod, 'bn', None)
            if isinstance(c, nn.Conv2d) and type(b) is nn.BatchNorm2d:
                mod.conv = fusion.fuse_conv_bn_eval(c.eval(), b.eval())
                mod.bn = nn.Identity()
                n[0] += 1
        rec(model)
        return n[0]
    #
    class CamHook:
        def __init__(self, model, layer):
            self.a = self.g = None
            self.h = dict(model.named_modules())[layer].register_forward_hook(self._f)
        def _f(self, m, i, o):
            self.a = o
            if o.requires_grad:
                o.register_hook(self._b)
        def _b(self, g):
            self.g = g
        def close(self):
            try:
                self.h.remove()
            finally:
                self.a = self.g = None
    #
    def _norm01(t):
        mn = t.amin(dim=(1, 2), keepdim=True)
        mx = t.amax(dim=(1, 2), keepdim=True)
        return (t - mn) / torch.clamp(mx - mn, min=1e-12)
    #
    def to_grid(t, g=IMG):
        if t.dim() == 2:
            t = t[None]
        t = F.interpolate(t[:, None].float(), size=(g, g), mode='bilinear', align_corners=False)[:, 0]
        return _norm01(t).detach().cpu().numpy()
    #
    def cam_maps(model, layer, x, tgt, plusplus=False):
        hk = CamHook(model, layer)
        try:
            model.zero_grad(set_to_none=True)
            x = x.to(DEV)
            out = model(x)
            sc = out.gather(1, tgt.view(-1, 1).to(DEV)).sum()
            sc.backward()
            A, G = hk.a, hk.g
            if A is None or G is None:
                raise RuntimeError('CAM hook captured no gradient')
            A, G = A.float(), G.float()
            if not plusplus:
                w = G.mean(dim=(2, 3), keepdim=True)
            else:
                G2 = G * G
                G3 = G2 * G
                den = 2.0 * G2 + A.sum(dim=(2, 3), keepdim=True) * G3
                alpha = torch.where(den.abs() > 1e-12, G2 / den, torch.zeros_like(den))
                w = (alpha * F.relu(G)).sum(dim=(2, 3), keepdim=True)
            cam = F.relu((w * A).sum(dim=1))
            return to_grid(cam), out.detach()
        finally:
            hk.close()
            model.zero_grad(set_to_none=True)
    #
    def layer_shapes(model, arch):
        from collections import OrderedDict
        shapes, hooks = OrderedDict(), []
        def mk(n):
            def fn(m, i, o):
                if torch.is_tensor(o):
                    shapes[n] = tuple(o.shape)
            return fn
        for n, m in model.named_modules():
            if n:
                hooks.append(m.register_forward_hook(mk(n)))
        model.eval()
        with torch.no_grad():
            model(torch.zeros(1, 3, IMG, IMG, device=next(model.parameters()).device))
        for h in hooks:
            h.remove()
        return shapes
    #
    def resolve_cam_layer(model, arch, min_spatial=4):
        sh = layer_shapes(model, arch)
        for cand in PREFERRED.get(arch, []):
            s = sh.get(cand)
            if s and len(s) == 4 and s[2] >= min_spatial and s[3] >= min_spatial:
                return cand
        best = None
        for n, s in sh.items():
            if len(s) == 4 and s[2] >= min_spatial and s[3] >= min_spatial:
                best = n
        if best is None:
            raise RuntimeError('%s: no 4-D feature map' % arch)
        return best
    #
    def topk_mask(h, k=K_MAIN):
        flat = np.asarray(h, np.float64).ravel()
        n = flat.size
        m = max(1, int(round(k * n)))
        order = np.lexsort((np.arange(n), -flat))
        mask = np.zeros(n, bool)
        mask[order[:m]] = True
        return mask.reshape(np.shape(h))
    #
    def _spearman_np(a, b):
        a = np.asarray(a, float).ravel()
        b = np.asarray(b, float).ravel()
        if a.size < 3 or np.std(a) < 1e-12 or np.std(b) < 1e-12:
            return float('nan')
        ra, rb = _rankdata(a), _rankdata(b)
        ra = ra - ra.mean()
        rb = rb - rb.mean()
        den = math.sqrt(float((ra * ra).sum()) * float((rb * rb).sum()))
        return float((ra * rb).sum() / den) if den > 0 else float('nan')
    #
    def is_collapsed(h):
        a = np.asarray(h, np.float64)
        return bool((np.unique(a).size < 2) or (float(a.std()) < 1e-6))
    #
    def build_fp32(arch):
        m = timm.create_model(arch, pretrained=False, num_classes=NCLS)
        sd = torch.load(ckpts[arch], map_location='cpu')
        if isinstance(sd, dict):
            for key in ['state_dict', 'model', 'model_state_dict']:
                if key in sd and isinstance(sd[key], dict):
                    sd = sd[key]
                    break
        sd = {k.replace('module.', ''): v for k, v in sd.items()}
        missing, unexpected = m.load_state_dict(sd, strict=False)
        if len(missing) > 8:
            raise RuntimeError('%s: %d missing keys - checkpoint mismatch' % (arch, len(missing)))
        return m.to(DEV).eval()
    #
    def make_fq(arch, calib_x):
        import torch.nn.utils.parametrize as P
        m = build_fp32(arch)
        ref = torch.randn(2, 3, IMG, IMG, device=DEV)
        with torch.no_grad():
            y0 = m(ref).float().clone()
        import copy
        bak = copy.deepcopy(m.state_dict())
        try:
            fold_conv_bn(m)
            with torch.no_grad():
                y1 = m(ref).float()
            if not torch.allclose(y0, y1, atol=2e-2, rtol=2e-2):
                m = build_fp32(arch)
                m.load_state_dict(bak)
                m.eval()
        except Exception:
            m = build_fp32(arch)
            m.load_state_dict(bak)
            m.eval()
        for n, mod in m.named_modules():
            if isinstance(mod, (nn.Conv2d, nn.Linear)):
                P.register_parametrization(mod, 'weight', WeightFQ())
        fqs = {}
        for n, mod in m.named_modules():
            if not n:
                continue
            if isinstance(mod, ACT_T) or (isinstance(mod, nn.Linear)
                                          and 'classifier' not in n and 'fc' not in n):
                fq = ActFQ().to(DEV)
                fqs[n] = fq
                mod.register_forward_hook(lambda M, I, O, _f=fq: _f(O))
        fq_in = ActFQ().to(DEV)
        fqs['__input__'] = fq_in
        m.register_forward_pre_hook(lambda M, I, _f=fq_in: (_f(I[0]),) + tuple(I[1:]))
        for f in fqs.values():
            f.observing = True
        with torch.no_grad():
            for i in range(0, len(calib_x), 32):
                m(calib_x[i:i + 32].to(DEV))
        for f in fqs.values():
            f.observing = False
        return m.eval()
    #
    rng = np.random.RandomState(SEED)
    cal_idx = []
    for c in sorted(TRAIN_DF.label.unique()):
        idx = TRAIN_DF.index[TRAIN_DF.label == c].values.copy()
        rng.shuffle(idx)
        cal_idx.extend(idx[:52].tolist())
    # original recipe (CELL 2, line 905): sklearn stratified 512 draw from TRAIN
    _ci, _ = train_test_split(np.arange(len(TRAIN_DF)), train_size=512,
                              random_state=SEED, stratify=TRAIN_DF.y.values)
    CALX = torch.from_numpy(load_batch(TRAIN_DF.iloc[sorted(_ci)].path.tolist()))
    log('fake-quant calibration tensor: %s' % (tuple(CALX.shape),))
    #
    def cam_pass(df, arch, model_fp, model_fq, layer, bs=24):
        recs = []
        paths = df.path.tolist()
        names = df.image.tolist()
        ys = df.y.values
        for i in range(0, len(paths), bs):
            xb = torch.from_numpy(load_batch(paths[i:i + bs]))
            yb = torch.as_tensor(ys[i:i + bs], dtype=torch.long)
            for plus in [False, True]:
                cf, of = cam_maps(model_fp, layer, xb, yb, plusplus=plus)
                cq, oq = cam_maps(model_fq, layer, xb, yb, plusplus=plus)
                pf = of.argmax(1).cpu().numpy()
                pq = oq.argmax(1).cpu().numpy()
                for j in range(len(cf)):
                    mf, mq = topk_mask(cf[j]), topk_mask(cq[j])
                    inter = float(np.logical_and(mf, mq).sum())
                    union = float(np.logical_or(mf, mq).sum())
                    ssz = float(mf.sum() + mq.sum())
                    recs.append(dict(
                        arch=arch, xai=('gradcampp' if plus else 'gradcam'),
                        image=names[i + j], y=int(ys[i + j]),
                        fp32_pred=int(pf[j]), int8_pred=int(pq[j]),
                        fp32_correct=int(pf[j] == ys[i + j]),
                        pred_match=int(pf[j] == pq[j]),
                        topk_iou=(inter / union if union else float('nan')),
                        topk_dice=(2.0 * inter / ssz if ssz else float('nan')),
                        spearman=_spearman_np(cf[j], cq[j]),
                        collapsed_fp32=int(is_collapsed(cf[j])),
                        collapsed_int8=int(is_collapsed(cq[j]))))
        return pd.DataFrame(recs)
    #
    all_rows = []
    gate_rows = []
    for arch in [a for a in ARCHS if a in ckpts]:
        if time.time() - t0 > BUDGET_S:
            log('   time budget reached; stopping TASK E early (partial results kept)')
            break
        try:
            mfp = build_fp32(arch)
            layer = resolve_cam_layer(mfp, arch)
            mfq = make_fq(arch, CALX)
            log('   %s CAM layer=%s' % (arch, layer))
            # --- GATE: reproduce the published 320-subset value first
            sub = PADDY[PADDY.image.isin(EVAL320)].reset_index(drop=True) if EVAL320 else VAL_DF.head(320)
            g = cam_pass(sub, arch, mfp, mfq, layer)
            for xai in ['gradcam', 'gradcampp']:
                gg = g[g.xai == xai]
                pub = float('nan')
                if P_DRIFT:
                    dd = pd.read_csv(P_DRIFT)
                    sel = dd[(dd.arch == arch) & (dd.sim == 'qdq') & (dd.xai == xai)
                             & np.isclose(dd.k.astype(float), K_MAIN)]
                    if len(sel):
                        pub = float(sel.topk_iou.mean())
                gate_rows.append(dict(arch=arch, xai=xai, n=len(gg),
                                      reproduced_iou=round(float(gg.topk_iou.mean()), 4),
                                      published_iou=round(pub, 4) if pub == pub else float('nan'),
                                      abs_delta=round(abs(float(gg.topk_iou.mean()) - pub), 4)
                                      if pub == pub else float('nan')))
                log('      GATE %-10s reproduced=%.4f published=%.4f'
                    % (xai, gate_rows[-1]['reproduced_iou'],
                       gate_rows[-1]['published_iou']))
            worst = np.nanmax([r['abs_delta'] for r in gate_rows if r['arch'] == arch])
            if not (worst == worst) or worst > 0.05:
                log('      GATE FAILED for %s (delta=%s) -> not extending to full split' % (arch, worst))
                continue
            full = cam_pass(VAL_DF, arch, mfp, mfq, layer)
            full['scope'] = 'full_val_split'
            all_rows.append(full)
            log('      full split done: %d rows' % len(full))
            del mfp, mfq
            torch.cuda.empty_cache()
        except Exception as e3:
            log('   %s TASK E error: %s: %s' % (arch, type(e3).__name__, e3))
    G = pd.DataFrame(gate_rows)
    if len(G):
        save('N26_fullsplit_cam_gate', G)
    assert all_rows, 'no architecture passed the reproduction gate'
    RAW = pd.concat(all_rows, ignore_index=True)
    save('RAW_fullsplit_cam', RAW)
    agg = RAW.groupby(['arch', 'xai']).agg(
        n=('topk_iou', 'size'),
        iou=('topk_iou', 'mean'),
        dice=('topk_dice', 'mean'),
        rho=('spearman', 'mean'),
        agreement=('pred_match', 'mean'),
        collapse_rate=('collapsed_int8', 'mean')).reset_index().round(4)
    save('N26_fullsplit_cam', agg)
    record('E_fullsplit_cam', 'OK', '%d archs, %d rows (%.1fs)'
           % (RAW.arch.nunique(), len(RAW), time.time() - t0))
    print(agg.to_string(index=False))
except Exception as e:
    record('E_fullsplit_cam', 'SKIPPED', '%s: %s' % (type(e).__name__, e))
#
#
# ===========================================================================
# FINAL STATUS
# ===========================================================================
S = pd.DataFrame(STATUS)
save('N00_rev3_status', S)
print()
print('=' * 78)
print('CELL 23 COMPLETE in %.1f s' % (time.time() - T_START))
print('=' * 78)
print(S.to_string(index=False))
print()
print('Files written to %s:' % OUT)
for f in sorted(os.listdir(OUT)):
    fp = os.path.join(OUT, f)
    if os.path.isfile(fp):
        print('   %-42s %8.1f KB' % (f, os.path.getsize(fp) / 1024.0))
ok = int((S.status == 'OK').sum())
print()
print('%d/%d tasks OK. Tasks marked SKIPPED did not error the notebook.' % (ok, len(S)))

onnxruntime available: True
[    7.6s] search roots: ['/kaggle/input', '/kaggle/working']
[    7.7s] RAW_qat_drift      : /kaggle/input/datasets/sunzilkhandaker/quantxai-endgame/tables/RAW_qat_drift.csv
[    7.7s] RAW_drift_all      : /kaggle/input/datasets/sunzilkhandaker/quantxai-endgame/tables/RAW_drift_all.csv
[    7.7s] T8_qat_mitigation  : /kaggle/input/datasets/sunzilkhandaker/quantxai-endgame/tables/T8_qat_mitigation.csv
[    7.7s] N10_lambda_sweep   : /kaggle/input/datasets/sunzilkhandaker/quantxai-endgame/tables/N10_lambda_sweep.csv
[    7.7s] N3_calib_ablation  : /kaggle/input/datasets/sunzilkhandaker/quantxai-endgame/tables/N3_calibration_ablation.csv
[    7.7s] T5_pred_agreement  : /kaggle/input/datasets/sunzilkhandaker/quantxai-endgame/tables/T5_prediction_agreement.csv
[    9.1s] paddy: 10407 images / 10 classes @ /kaggle/input/datasets/imbikramsaha/paddy-doctor/paddy-disease-classification/train_images
[    9.1s] split: train=8325 val=2082  (paper: 8325 / 2082)
[    9.3